In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

# evaluation imports
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_json("data/telemetry_data.jsonl", lines=True)
df.head()

In [ ]:
X = df[[
    "cpu_usage",
    "memory_usage",
    "disk_io",
    "network_latency"
]]
y = df["recommended_action"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
simple_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=2,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

complex_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42
)

In [ ]:
simple_model.fit(X_train, y_train)
complex_model.fit(X_train, y_train)

In [ ]:
y_pred_simple = simple_model.predict(X_test)
y_pred_complex = complex_model.predict(X_test)

# ACCURACY

In [ ]:
simple_model_acc = accuracy_score(y_test, y_pred_simple)
complex_model_acc = accuracy_score(y_test, y_pred_complex)

print(f"Simple model accuracy: {simple_model_acc}")
print(f"Complex model accuracy: {complex_model_acc}")

# FEATURE IMPORANCES

### Simple Model

In [ ]:
importances = simple_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10,6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.xlabel("Feature importance")
plt.title("Feature importance in log error classification")
plt.show()

### Complex Model

In [ ]:
importances = complex_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10,6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.xlabel("Feature importance")
plt.title("Feature importance in log error classification")
plt.show()

# CONFUSION MATRIX

### Simple Model

In [ ]:
cmat = confusion_matrix(y_test, y_pred_simple)
plt.figure(figsize=(12, 8))
sns.heatmap(cmat, annot=True, fmt='d', cmap='Blues', 
            xticklabels=simple_model.classes_, 
            yticklabels=simple_model.classes_)

plt.title('Confusion Matrix: Predicted vs Actual')
plt.ylabel('Actual Action')
plt.xlabel('Predicted Action')
plt.xticks(rotation=45, ha='right')
plt.show()

### Complex Model

In [ ]:
cmat = confusion_matrix(y_test, y_pred_complex)
plt.figure(figsize=(12, 8))
sns.heatmap(cmat, annot=True, fmt='d', cmap='Blues', 
            xticklabels=complex_model.classes_, 
            yticklabels=complex_model.classes_)

plt.title('Confusion Matrix: Predicted vs Actual')
plt.ylabel('Actual Action')
plt.xlabel('Predicted Action')
plt.xticks(rotation=45, ha='right')
plt.show()

# PRECISION VS RECALL

### Simple Model

In [ ]:
report = classification_report(y_test, y_pred_simple, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.iloc[:-3, :2].plot(kind='bar', figsize=(12, 5))
plt.title('Precision vs Recall per Recommended Action')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower left')
plt.show()

### Complex Model

In [ ]:
report = classification_report(y_test, y_pred_complex, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.iloc[:-3, :2].plot(kind='bar', figsize=(12, 5))
plt.title('Precision vs Recall per Recommended Action')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='lower left')
plt.show()

# TREES

### Simple Model

In [ ]:
plt.figure(figsize=(100,20))

plot_tree(
    simple_model,
    feature_names=X.columns,
    class_names=simple_model.classes_,
    filled=True,
    proportion=True,
    fontsize=9,
    precision=1
)

plt.tight_layout()
plt.show()

### Complex Model

In [ ]:
plt.figure(figsize=(100,20))

plot_tree(
    complex_model,
    feature_names=X.columns,
    class_names=complex_model.classes_,
    filled=True,
    proportion=True,
    fontsize=9,
    precision=1
)

plt.tight_layout()
plt.show()

# Save models

In [ ]:
import joblib

joblib.dump(simple_model, "experiment/models/simple_model.pkl")
joblib.dump(complex_model, "experiment/models/complex_model.pkl")

print("Saved models!")